In [ ]:
! pip install pandas requests python-dotenv openpyxl xlrd jupyterlab

In [1]:
import pandas as pd
import os

RAW_DIR = "../data/raw/"
STAGING_DIR = "../data/staging/"
os.makedirs(STAGING_DIR, exist_ok=True)

# Negara yang kita fokuskan
TARGET_COUNTRIES = [
    "United States", "China", "Germany", 
    "Japan", "India", "United Kingdom",
    "France", "Brazil"
]

print("Setup selesai ✓")

Setup selesai ✓


In [2]:
def read_worldbank_excel(filename, indicator_name):
    """
    Baca file Excel World Bank dan kembalikan dataframe yang sudah dirapikan.
    Format World Bank: baris 1-3 metadata, kolom A = negara, kolom sisanya = periode.
    """
    filepath = os.path.join(RAW_DIR, filename)
    
    # Baca dulu tanpa skip untuk lihat struktur
    df_raw = pd.read_excel(filepath, header=None)
    
    # Cari baris yang berisi header (biasanya row ke-3 atau ke-4)
    # Tandanya: ada teks 'Country Name' atau nama negara pertama
    header_row = None
    for i, row in df_raw.iterrows():
        if any(str(v).strip() in ["Country Name", "Country", ""] == False 
               and str(v).strip() != "nan" 
               for v in row[:2]):
            # Cek apakah kolom lain berisi tahun/periode
            non_empty = [v for v in row if str(v).strip() not in ["nan", ""]]
            if len(non_empty) > 5:
                header_row = i
                break
    
    # Baca ulang dengan header yang benar
    df = pd.read_excel(filepath, header=header_row)
    
    # Set index ke nama negara (kolom pertama)
    df = df.set_index(df.columns[0])
    
    # Filter hanya negara yang kita butuhkan
    available = [c for c in TARGET_COUNTRIES if c in df.index]
    df = df.loc[available]
    
    # Filter kolom periode 2022-2024 saja
    period_cols = []
    for col in df.columns:
        col_str = str(col)
        # Format bisa: 2022M01, Jan-2022, 2022-01, 2022Q1, atau integer tahun
        if any(str(y) in col_str for y in ["2022", "2023", "2024"]):
            period_cols.append(col)
    
    df = df[period_cols]
    
    # Transpose: jadikan negara sebagai kolom, periode sebagai baris
    df = df.T
    df.index.name = "period_raw"
    df = df.reset_index()
    
    # Melt ke format long (tidy)
    df_long = df.melt(
        id_vars="period_raw",
        var_name="country",
        value_name=indicator_name
    )
    
    print(f"  ✓ {filename[:40]}... → {df_long.shape[0]} rows, negara: {df_long['country'].unique().tolist()}")
    return df_long

In [4]:
print("Membaca file Excel World Bank...\n")

wb_datasets = {}

# 1. CPI
wb_datasets["cpi"] = read_worldbank_excel(
    "CPI Price, % y-o-y, nominal, seas. adj..xlsx",
    "cpi_pct_yoy"
)

# 2. GDP
wb_datasets["gdp"] = read_worldbank_excel(
    "GDP at market prices, current US$, millions, seas. adj..xlsx",
    "gdp_current_usd_mn"
)

# 3. Industrial Production
wb_datasets["industrial"] = read_worldbank_excel(
    "Industrial Production, constant 2010 US$, seas. adj..xlsx",
    "industrial_prod_usd"
)

# 4. Unemployment
wb_datasets["unemployment"] = read_worldbank_excel(
    "Unemployment Rate, seas. adj..xlsx",
    "unemployment_rate"
)

# 5. Exchange Rate
wb_datasets["exchange_rate"] = read_worldbank_excel(
    "Exchange rate, new LCU per USD extended backward, period average.xlsx",
    "exchange_rate_lcu_usd"
)

Membaca file Excel World Bank...

  ✓ CPI Price, % y-o-y, nominal, seas. adj..... → 0 rows, negara: []
  ✓ GDP at market prices, current US$, milli... → 0 rows, negara: []
  ✓ Industrial Production, constant 2010 US$... → 0 rows, negara: []
  ✓ Unemployment Rate, seas. adj..xlsx... → 0 rows, negara: []
  ✓ Exchange rate, new LCU per USD extended ... → 0 rows, negara: []


In [5]:
for name, df in wb_datasets.items():
    out_path = os.path.join(STAGING_DIR, f"wb_{name}_2022_2024.csv")
    df.to_csv(out_path, index=False)
    print(f"✓ Tersimpan: wb_{name}_2022_2024.csv ({df.shape[0]} rows)")

print("\nSemua file World Bank berhasil disimpan ke staging ✓")

✓ Tersimpan: wb_cpi_2022_2024.csv (0 rows)
✓ Tersimpan: wb_gdp_2022_2024.csv (0 rows)
✓ Tersimpan: wb_industrial_2022_2024.csv (0 rows)
✓ Tersimpan: wb_unemployment_2022_2024.csv (0 rows)
✓ Tersimpan: wb_exchange_rate_2022_2024.csv (0 rows)

Semua file World Bank berhasil disimpan ke staging ✓


In [6]:
# Cek salah satu hasilnya
print("Preview CPI dataset:")
print(wb_datasets["cpi"].head(10))
print("\nTipe data:")
print(wb_datasets["cpi"].dtypes)

Preview CPI dataset:
Empty DataFrame
Columns: [period_raw, country, cpi_pct_yoy]
Index: []

Tipe data:
period_raw       int64
country         object
cpi_pct_yoy    float64
dtype: object
